## Detailed Algorithm Demonstration

### Import libs

In [ ]:
from dataclasses import asdict

from pyproj import Transformer
import cv2
import numpy as np
import matplotlib.pyplot as plt
from lightglue import viz2d

from uavcalibration.datasets import VisLocDataset
from uavcalibration.transform import *
import uavcalibration.rectification as rect
from uavcalibration.matching import *

dataset = VisLocDataset(r"..\datasets\UAV_VisLoc_example")
uav_data = dataset[0]

### Rectify uav image

In [ ]:
raw_shape = uav_data.uav_image.shape[1], uav_data.uav_image.shape[0]

camera_mat = rect.camera_mat(raw_shape, uav_data.focal_length)
rotate_mat = rect.rotate_mat(yaw=uav_data.yaw, pitch=uav_data.pitch, roll=uav_data.roll)
rect_mat = rect.rectify_mat(camera_mat=camera_mat, rotate_mat=rotate_mat)
assert uav_data.height
crs_trans = rect.crs_trans(
    longitude=uav_data.longitude,
    latitude=uav_data.latitude,
    camera_mat=camera_mat,
    rotate_mat=rotate_mat,
    height=uav_data.height,
)
transform = Transform(
    pix_mat=rect_mat,
    src_shape=raw_shape,
    crs=crs_trans,
)
transform.adjust_shape()

image0 = transform.warp(uav_data.uav_image)
image1 = uav_data.satellite_image

### match uav image with satellite image

In [ ]:
match_result = match_images(
    image0,
    image1,
    method=MatchingMethod.LIGHTGLUE,
    max_kpts0=2048,
    max_kpts1=2048,
)
plot_matches(match_result, image0, image1)

### Use RANSAC to remove outliers and fit homography transform matrix

In [ ]:
resolution = transform.crs.resolution  # meters per pixel
tolerance = 5  # meters
threshold = tolerance / resolution  # tolerance threshold
homography_result = match_homography(match_result.kpts0, match_result.kpts1, threshold)
plot_matches(homography_result, image0, image1)

### Caculate total transform

In [ ]:
transform.follow(homography_result.mat)
transform.dst_shape = image1.shape[1], image1.shape[0]
transform.crs.mat = uav_data.satellite_transform
transform.crs.crs = uav_data.satellite_crs
print(transform.combined)